In [59]:
# Little test for the following madness
# Test the sum of sigma_+ matrices acting on either the first or second of two qubits
sigma_p = [0 1.0+0*im; 0 0]
sigma_m = [0 0; 1.0+0*im 0]
sigma = 0.5*(sigma_p + sigma_m)
eye = [1.0 0; 0 1.0+0*im]
sigma_1 = kron(sigma, eye, eye)
sigma_2 = kron(eye, sigma, eye)
sigma_3 = kron(eye, eye, sigma)
sigma_sum = sigma_1 + sigma_2 + sigma_3
# Start with an equal superposition of single excitation states 
phi0 = [0; 0; 0; 1.0; 0; 1.0; 1.0; 0*im]/sqrt(3)
#phi0 = [0; 0; 0; 0; 0; 0; 0; 1]
# do time evolution of phi0 via matrix exponential of sigma_p_sum	
phi = pi
sigma_exp = exp(-1.0*im*sigma_sum * phi)
phi1 = sigma_exp*phi0

8-element Vector{ComplexF64}:
 -3.5035087906569203e-16 + 0.0im
                     0.0 + 0.5773502691896257im
                     0.0 + 0.5773502691896258im
 -1.5152958558971173e-16 + 0.0im
                     0.0 + 0.5773502691896258im
 -2.1826232297723693e-16 + 0.0im
  -1.933574182975488e-16 + 0.0im
                     0.0 + 1.078853874379298e-17im

In [53]:
A = exp(-1.0*im*sigma*pi)

2×2 Matrix{ComplexF64}:
 1.50134e-16+0.0im          0.0-1.0im
         0.0-1.0im  1.50134e-16+0.0im

In [62]:
kron(sigma_p, sigma_m)-kron(sigma_m, sigma_p)

4×4 Matrix{ComplexF64}:
 0.0+0.0im   0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im   0.0+0.0im  1.0+0.0im  0.0+0.0im
 0.0+0.0im  -1.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im   0.0+0.0im  0.0+0.0im  0.0+0.0im

In [ ]:
include("initial_states.jl")

In [4]:
# This function can be made more elaborate to account for g and \Delta dependent states 
function single_excitation_state_gen(N::Int)::Function
    # Generates the state 1/\sqrt{N} sum_{i=1}^N \sigma_x^i |0...0>
    # where N is the number of spins
    # and n_z is the number of spins in the z-direction in the operator
    # Calculates <\prod_{i=1}^n \sigma_z^{k_i}>:
    function generator(spin_types::String)::ComplexF64
        op_exp::ComplexF64 = 0.0 + im * 0.0
        if occursin("x", spin_types) || occursin("y", spin_types)
        else
            # how many times does z occur?
            n_z::Int = count(c -> c == 'z', spin_types)
            op_exp += (-1)^n_z*(1-2*n_z/N)
        end 
        return op_exp
    end
    return generator
end

single_excitation_state_gen (generic function with 1 method)

In [5]:
function gibbs_state_gen(p::Float64)::Function
    # generator, that returns a function that computes the expectation value for a specified number of fcreation and annihilation operators
    # p is the occupation probability of the ground state
    # Theta is the phase of the coherent state
    x::Float64 = 0
    y::Float64 = 0
    z::Float64 = 1 - 2 * p
    function generator(spin_types::String)::ComplexF64
        # if any x or y in string return 0.0
        if occursin("x", spin_types) || occursin("y", spin_types)
            return 0.0 + im * 0.0
        else
            # how many times does z occur?
            z_occurrences::Int = count(c -> c == 'z', spin_types)
            return z^z_occurrences
        end
    end
    return generator
end

gibbs_state_gen (generic function with 1 method)

In [6]:
function coherent_state_gen(alpha::Float64, Theta::Float64)::Function
    # generator, that returns a function that computes the expectation value for a specified number of fcreation and annihilation operators
    # alpha is the amplitude of the coherent state
    # Theta is the phase of the coherent state
    abs_alpha::Float64 = abs(alpha)
    e_itheta::ComplexF64 = exp(im * Theta)
    function generator(cavity_types::String)::ComplexF64
        creation::Int = count(c -> c == '+', cavity_types)
        annihilation::Int = count(c -> c == '-', cavity_types)
        return abs_alpha^(creation + annihilation) * e_itheta^(creation - annihilation)
    end
    return generator
end

coherent_state_gen (generic function with 1 method)

In [7]:
function haskeys(dict::Dict, keys::Array)::Bool
    # checks if a dictionary has all the keys in the array
    for key in keys
        if !haskey(dict, key)
            return false
        end
    end
    return true
end
# generate arguments for a function from a dictionary and an argument_name list 
function generate_args(dict::Dict, arg_names::Array)::Array
    args::Array = []
    for arg_name in arg_names
        push!(args, dict[arg_name])
    end
    return args
end

generate_args (generic function with 1 method)

In [12]:

function term_str2states(inverse_op_index_vec, spin_type::String, cavity_type::String, spin_param::Dict, cavity_param::Dict)::Vector{ComplexF64}
    # Generate initial conditions for the equations
    # inverse_op_index_vec is a vector of strings of the form "++-+" (potentially with indexes)
    # spin_type is either "gibbs" or "single_excitation" (update with new functions)
    # cavity_type currently_only_supports "coherent" (update with new functions)
    initial_values::Vector{ComplexF64} = Vector{ComplexF64}(undef, length(inverse_op_index_vec))
    if cavity_type == "coherent"
        coherent_args::Vector{String} = ["alpha", "Theta"]
        if !haskeys(cavity_param, coherent_args)
            error("cavity_param must have keys alpha and Theta for coherent states")
        end
        cavity_fun = coherent_state_gen(generate_args(cavity_param, coherent_args)...)
    else
        error("cavity_type unknown")
    end
    if spin_type == "gibbs"
        if !haskey(spin_param, "p")
            error("spin_param must have key p for gibbs states")
        end
        spin_fun = gibbs_state_gen(spin_param["p"])
    elseif spin_type == "single_excitation"
        if !haskey(spin_param, "N")
            error("spin_param must have key N for single excitation states")
        end
        spin_fun = single_excitation_state_gen(spin_param["N"])
    else
        error("spin_type unknown")
    end
    for i in 1:length(inverse_op_index_vec)
        initial_values[i] = spin_fun(inverse_op_index_vec[i]) * cavity_fun(inverse_op_index_vec[i])
    end
    return initial_values
end
# Test 
term_str = ["z", "zz", "zzz", "zzzz"]
spin_param = Dict("N" => 10^6)
cavity_param = Dict("alpha" => 1.0, "Theta" => 0.0)
initial_values = term_str2states(term_str, "single_excitation", "coherent", spin_param, cavity_param)

4-element Vector{ComplexF64}:
 -0.999998 + 0.0im
  0.999996 + 0.0im
 -0.999994 + 0.0im
  0.999992 + 0.0im

In [16]:
z = initial_values[1]
zz = initial_values[2]
zzz = initial_values[3]
zzzz = initial_values[4]
println("Cumulants Accuracy")
println("zz: ", zz - z^2)
println("zzz: ", zzz - zz*z*3+2*z^3)

Cumulants Accuracy
zz: -4.000133557724439e-12 + 0.0im
zzz: 0.0 + 0.0im
